In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from IPython.core.interactiveshell import InteractiveShell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all" 

In [70]:
# 加载数据
train = pd.read_csv("train_new_features.csv")
test = pd.read_csv("test_new_features.csv")

y=train['Transported'].copy().astype(int)
X=train.drop('Transported', axis=1).copy()

# 合并测试集和训练集
data=pd.concat([X, test], axis=0).reset_index(drop=True)

# 缺失值分析

In [71]:
# 缺失值数量
print("\n缺失值数量")
print(data.isnull().sum())

# # 生成缺失值指示矩阵（1=缺失，0=存在）
# missing_indicator = data.isnull().astype(int)

# # 计算缺失值之间的相关性
# missing_corr = missing_indicator.corr()

# # 绘制热力图
# plt.figure(figsize=(15, 8))
# sns.heatmap(missing_corr, annot=True, cmap='coolwarm', fmt=".2f", 
#             mask=np.triu(np.ones_like(missing_corr, dtype=bool)))
# plt.title("Correlation Between Missing Values", fontsize=12)
# plt.show()


缺失值数量
Unnamed: 0        0
PassengerId       0
HomePlanet      288
CryoSleep       310
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Name            294
exp_tol           0
No_spending       0
Group             0
Group_size        0
Solo              0
Cabin_deck      299
Cabin_number    299
Cabin_side      299
Surname         294
Family_Size     294
dtype: int64


发现缺失值相关性很弱

In [72]:
initial_count = len(data)
data_dropped = data.dropna()
remaining_ratio = len(data_dropped) / initial_count
print(f"移除所有缺失行后剩余数据比例: {remaining_ratio:.2%}")

移除所有缺失行后剩余数据比例: 76.23%


剩余数据量太少，不能删

## 缺失值处理



## 1.总花费为0，乘客很可能是冬眠的

In [73]:
#查看冬眠与花费的关系
data.groupby(['No_spending','CryoSleep'])['CryoSleep'].size().unstack().fillna(0)


CryoSleep,False,True
No_spending,,
0,7339.0,0.0
1,740.0,4581.0


In [74]:
# Missing values before
CSL_bef=data['CryoSleep'].isna().sum()

# Fill missing values using the mode


data.loc[data['CryoSleep'].isna() & data['No_spending']==0,'CryoSleep']  = True
data.loc[data['CryoSleep'].isna() & data['No_spending']==1,'CryoSleep']  = False

# Print number of missing values left
print('#CryoSleep missing values before:',CSL_bef)
print('#CryoSleep missing values after:',data['CryoSleep'].isna().sum())

#CryoSleep missing values before: 310
#CryoSleep missing values after: 0


冬眠的乘客一定没消费

In [75]:
exp_feats=['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
E_bef=data[exp_feats].isna().sum().sum()

for col in exp_feats:
    data.loc[(data[col].isna()) & (data['CryoSleep']==True), col]=0

print('#Expenditure missing values before:',E_bef)
print('#Expenditure missing values after:',data[exp_feats].isna().sum().sum())

#Expenditure missing values before: 1410
#Expenditure missing values after: 13


## 2.同一组的人来自同一颗星球

In [83]:
GHP_gb=data.groupby(['Group','HomePlanet'])['HomePlanet'].size().unstack().fillna(0)
GHP_gb.head()
GHP_gb.groupby(level='Group').size().value_counts()


HomePlanet,Earth,Europa,Mars
Group,,,
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,2.0,0.0
4,1.0,0.0,0.0
5,1.0,0.0,0.0


1    9124
Name: count, dtype: int64

每个组的乘客都来自同一星球

In [ ]:
#填充缺失值

HP_bef=data['HomePlanet'].isna().sum()

# Passengers with missing HomePlanet and in a group with known HomePlanet
GHP_index=data[data['HomePlanet'].isna()][(data[data['HomePlanet'].isna()]['Group']).isin(GHP_gb.index)].index

# Fill corresponding missing values
data.loc[GHP_index,'HomePlanet']=data.iloc[GHP_index,:]['Group'].map(lambda x: GHP_gb.idxmax(axis=1)[x])

# Print number of missing values left
print('#HomePlanet missing values before:',HP_bef)
print('#HomePlanet missing values after:',data['HomePlanet'].isna().sum())





#HomePlanet missing values before: 288
#HomePlanet missing values after: 157
